# 01 · Chuẩn bị VNTS + train detector YOLO11n (1 lớp và 52 lớp)

Cần **GPU T4**, bật Internet. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input trên Kaggle:
- dataset `maitam/vietnamese-traffic-signs` (VNTS, 52 lớp đặt tên theo mã QCVN, CC BY-SA 4.0)
- dataset `vn-dashcam-vision-code` (code repo này)

Trên Colab: code clone từ GitHub, VNTS tải bằng kaggle CLI nên cần thêm Secrets (biểu tượng chìa khoá bên trái,
bật Notebook access) `KAGGLE_USERNAME`, `KAGGLE_KEY` (lấy ở kaggle.com → Settings → API).
Mọi output nằm trong `MyDrive/ai-portfolio/vn-dashcam-vision` để notebook 02-04 dùng lại.

Train 2 detector để so sánh:
- `1cls`: mọi biển gộp 1 lớp "sign" -> tầng 1 của pipeline 2 tầng
- `ncls`: 52 lớp -> baseline 1 tầng

Ultralytics là AGPL-3.0: repo public thì ổn, dùng thương mại phải mở mã hoặc mua license.

In [ ]:
import glob, json, os, sys

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


# Colab: đọc hàng nghìn ảnh từ Google Drive rất chậm -> dataset để ở ổ cục bộ /content (mất khi ngắt,
# chạy lại cell là có), còn model / kết quả / checkpoint YOLO thì nằm trong WORK trên Drive.
DATA = "/content/data" if IN_COLAB else "datasets"
DS = f"{DATA}/vnts"
os.makedirs(DATA, exist_ok=True)

# VNTS: Kaggle add dataset làm input; Colab tải bằng kaggle CLI về ổ cục bộ
if IN_KAGGLE:
    VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
else:
    VNTS = f"{DATA}/vnts_raw"
    if not glob.glob(f"{VNTS}/*"):
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            os.environ[k] = secret(k) or ""
        !kaggle datasets download -d maitam/vietnamese-traffic-signs -p {VNTS} --unzip
print(CODE, VNTS)
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m dashcam.prepare_vnts --src {VNTS} --out {DS}
print(json.dumps(json.load(open(f"{DS}/class_stats.json"))["train"], ensure_ascii=False)[:1500])

## Phân bố lớp (lệch rất mạnh)

In [ ]:
import matplotlib.pyplot as plt

st = json.load(open(f"{DS}/class_stats.json"))["train"]
items = sorted(st.items(), key=lambda x: -x[1])
plt.figure(figsize=(16, 4))
plt.bar([k for k, _ in items], [v for _, v in items])
plt.xticks(rotation=90, fontsize=7); plt.yscale("log"); plt.title("số crop train mỗi lớp (log)")
plt.tight_layout(); plt.savefig("class_dist.png", dpi=110)

## Copy-paste cho lớp hiếm (< 50 mẫu train)

In [ ]:
!python -m dashcam.synth_rare --data {DS} --rare 50 --per-class 150

## Train YOLO11n
imgsz 640: biển xa chỉ ~15px nên 960 sẽ tốt hơn cho recall, nhưng chạy CPU chậm ~2.2 lần.
Giữ 640 vì mục tiêu là chạy được trên laptop; ghi lại đánh đổi này trong NOTES.

In [ ]:
from ultralytics import YOLO

for name in ("1cls", "ncls"):
    m = YOLO("yolo11n.pt")
    m.train(data=f"{DS}/yolo_{name}/data.yaml", imgsz=640, epochs=60, patience=15, batch=32,
            fliplr=0.0,  # KHÔNG lật ngang: cấm rẽ trái <-> cấm rẽ phải
            mosaic=1.0, degrees=5, hsv_v=0.5, project="runs", name=f"det_{name}", exist_ok=True, verbose=False)

In [ ]:
for name in ("1cls", "ncls"):
    m = YOLO(f"runs/det_{name}/weights/best.pt")
    r = m.val(data=f"{DS}/yolo_{name}/data.yaml", split="test", imgsz=640, verbose=False)
    print(name, "mAP50", round(r.box.map50, 4), "mAP50-95", round(r.box.map, 4))
    m.export(format="onnx", imgsz=640, opset=17, simplify=True)

In [ ]:
!mkdir -p models && cp runs/det_1cls/weights/best.onnx models/detector_1cls.onnx && cp runs/det_ncls/weights/best.onnx models/detector_ncls.onnx
!cp {DS}/names.json models/ && ls -la models
# giữ crops + class_stats cho notebook 02
!tar czf crops.tgz -C {DATA} vnts/crops vnts/names.json vnts/class_stats.json